# Pilot: Perception & Reasoning Entropy Signal Check (FERMAT + Qwen2.5-VL)

Purpose: the smallest, fastest end-to-end run to check whether perception entropy
(instability in transcribing handwritten math) and reasoning entropy (instability in
grading an answer for errors) are visibly higher on items the model gets wrong. This
is a scoped-down pilot, not the full study — no bootstrap CIs, no baseline suite, no
conformal calibration, no human double grading.

**This notebook does not carry its own copy of the pipeline logic.** Every non-GPU
piece (data loading/filtering, prompt formatting, output parsing, entropy calculation)
lives in the `pilot/` package and is unit-tested locally. This notebook clones that
same code repo fresh each session and installs it, so there is a single source of
truth — editing a module locally and re-running this notebook next session picks up
the change automatically, with no manual copy/paste re-sync step.

Only the GPU-dependent parts live here: installing GPU deps, loading the model, and
running the sampling loop.

In [1]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 64.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 79.1 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code/results access cell.
import json
import os
from getpass import getpass

from huggingface_hub import login

# --- Drive mount first: it holds both the model cache and the token store ---
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Tokens: entered ONCE, then cached on your Drive ---
# Deliberately not hardcoded in this notebook. This file is tracked in a
# public repo, and GitHub's secret scanning auto-revokes any ghp_ token that
# lands in a public commit -- so an inline token would stop working by
# itself. Drive is private to your account, survives runtime recycling, and
# git never touches it, so you get the same "no retyping" result safely.
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False  # set True once to replace previously saved tokens


def get_token(name, prompt):
    """Return a saved token, prompting (once) and persisting it if absent."""
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

# --- Clone the repo (code + results live in the same repo for this pilot) ---
# Cloned anonymously: the repo is public, so read access needs no token, and
# keeping the token out of the clone URL means a clone error can never echo
# it into this notebook's saved output. The token is used only to push.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone -q {REPO_URL} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [3]:
# Model load cell.
# Start with the 3B model for the first smoke test -- same prompt format and
# code path as the 7B, but noticeably faster to load and run, so early bugs
# get caught cheaply. Swap MODEL_ID to the 7B line below once the pipeline
# runs cleanly end to end on the 3B.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # swap in once the 3B pipeline is clean

# If memory is tight on the 7B, load in 4-bit instead:
# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     quantization_config=quantization_config,
#     device_map="auto",
#     cache_dir=DRIVE_MODEL_CACHE,
# )

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=DRIVE_MODEL_CACHE,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [ ]:
# Sampling loop cell.
#
# K=5 samples at temperature=0.7 for each of the transcription and grading
# prompts, plus 2 samples at temperature=0 (greedy, back-to-back) as a
# low-temperature sanity anchor -- 2 draws, not 1, because a single-sample
# entropy is mathematically zero by definition and could never catch a real
# bug (e.g. an accidental sampling flag, batching nondeterminism). With 2
# draws it's a genuine, if noisy, instability check.
#
# Each generation call is wrapped in a bounded retry scoped strictly to
# infrastructure-level failures (dropped connection, transient OOM, Colab
# runtime hiccup) -- never around parsing. A response that fails to parse is
# real data about that sample's behavior under that prompt, not a transient
# failure to retry past; retrying until a clean parse appears would silently
# bias every entropy estimate downward.
#
# Results are checkpointed to Drive as completed items, plus a partial-item
# checkpoint every 200 generation calls. Re-run this cell after a disconnect
# and it resumes from the last checkpoint instead of starting over.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

# This loop makes 14 model calls per item (2 prompts x (5 sampled + 2 greedy)),
# so N=100 is ~1400 calls and can take a while on a T4.
N = 100
SEED = 42
K = 5
TEMP = 0.7
N_TEMP0 = 2
MAX_RETRIES = 3
RETRY_PAUSE_SECONDS = 5
CHECKPOINT_EVERY_GENERATIONS = 200

# Only these fields are needed downstream (cell 6 never touches the image),
# and dropping the PIL image keeps each checkpoint row JSON-serializable.
META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")

INFRA_EXCEPTIONS = (
    ConnectionError,
    TimeoutError,
    torch.cuda.OutOfMemoryError,
    OSError,
)


def generate(messages, do_sample: bool, temperature: float | None):
    """Run one generation call, retrying only on infrastructure-level failures."""
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            text_prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text_prompt],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(model.device)

            gen_kwargs = {"max_new_tokens": 512, "do_sample": do_sample}
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.no_grad():
                output_ids = model.generate(**inputs, **gen_kwargs)

            trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
            return processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
        except INFRA_EXCEPTIONS as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_PAUSE_SECONDS)
    raise last_exc


def atomic_json_dump(obj, path):
    """Write JSON through a temp file so a disconnect cannot corrupt it."""
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f)
    os.replace(tmp_path, path)


sample = pilot.data.load_fermat_sample(n=N, seed=SEED)
n_items = len(sample)
calls_per_item = 2 * (K + N_TEMP0)

# Checkpoint files are keyed by model/N/seed so a different config starts fresh
# rather than silently resuming from an unrelated run.
CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_prefix = f"raw_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}"
completed_path = f"{CHECKPOINT_DIR}/{checkpoint_prefix}.jsonl"
partial_path = f"{CHECKPOINT_DIR}/{checkpoint_prefix}.partial.json"
checkpoint_config = {
    "model_id": MODEL_ID,
    "n": N,
    "seed": SEED,
    "k": K,
    "temp": TEMP,
    "n_temp0": N_TEMP0,
}



def entry_is_complete(entry):
    return (
        len(entry.get("transcription_samples_raw", [])) == K
        and len(entry.get("grading_samples_raw", [])) == K
        and len(entry.get("transcription_temp0_raw", [])) == N_TEMP0
        and len(entry.get("grading_temp0_raw", [])) == N_TEMP0
    )


def entry_matches_sample(entry, sample_item):
    saved_item = entry.get("item", {})
    return all(saved_item.get(k) == sample_item[k] for k in META_FIELDS)


def rewrite_completed_checkpoint(entries):
    with open(completed_path, "w") as f:
        for entry in entries:
            f.write(json.dumps(entry, default=str) + "\n")
        f.flush()


def validate_completed_checkpoint(entries):
    valid_entries = []
    for idx, entry in enumerate(entries[:n_items]):
        if not entry_is_complete(entry):
            print(f"Checkpoint item {idx + 1} is incomplete; resuming from there.")
            break
        if not entry_matches_sample(entry, sample[idx]):
            print(f"Checkpoint item {idx + 1} does not match expected sample order; resuming from there.")
            break
        valid_entries.append(entry)

    if len(valid_entries) != len(entries):
        print(
            f"Truncating completed checkpoint from {len(entries)} to "
            f"{len(valid_entries)} valid items."
        )
        rewrite_completed_checkpoint(valid_entries)
    return valid_entries


raw_results = []
if os.path.exists(completed_path):
    with open(completed_path) as f:
        raw_results = [json.loads(line) for line in f if line.strip()]
    print(f"Found completed-item checkpoint: {len(raw_results)} items")
    print(f"  {completed_path}")
    raw_results = validate_completed_checkpoint(raw_results)
    print(f"Validated completed items: {len(raw_results)}")

partial_state = None
if os.path.exists(partial_path):
    with open(partial_path) as f:
        candidate = json.load(f)
    if (
        candidate.get("config") == checkpoint_config
        and candidate.get("completed_items") == len(raw_results)
        and candidate.get("item_idx", -1) >= len(raw_results)
    ):
        partial_state = candidate
        print(
            "Found partial checkpoint: "
            f"item {partial_state['item_idx'] + 1}/{n_items}"
        )
    else:
        print("Ignoring stale partial checkpoint that does not match completed items/config.")


def count_entry_calls(entry):
    return sum(
        len(entry.get(field, []))
        for field in (
            "transcription_samples_raw",
            "grading_samples_raw",
            "transcription_temp0_raw",
            "grading_temp0_raw",
        )
    )


calls_since_partial_checkpoint = 0


def save_partial_checkpoint(item_idx, entry):
    atomic_json_dump(
        {
            "config": checkpoint_config,
            "completed_items": len(raw_results),
            "item_idx": item_idx,
            "entry": entry,
        },
        partial_path,
    )


def maybe_save_partial_checkpoint(item_idx, entry):
    global calls_since_partial_checkpoint
    calls_since_partial_checkpoint += 1
    if calls_since_partial_checkpoint >= CHECKPOINT_EVERY_GENERATIONS:
        save_partial_checkpoint(item_idx, entry)
        calls_since_partial_checkpoint = 0


def run_batch(messages, n, do_sample, temperature, stage, item_idx, outputs, entry, pbar):
    """Draw missing samples for one prompt, resuming from existing outputs."""
    for j in range(len(outputs), n):
        pbar.set_postfix_str(f"item {item_idx + 1}/{n_items} | {stage} {j + 1}/{n}")
        print(f"item {item_idx + 1}/{n_items} | {stage} {j + 1}/{n}", flush=True)
        outputs.append(generate(messages, do_sample=do_sample, temperature=temperature))
        maybe_save_partial_checkpoint(item_idx, entry)
        pbar.update(1)
    return outputs


n_done = len(raw_results)
partial_done_calls = 0
if partial_state is not None:
    partial_done_calls = count_entry_calls(partial_state["entry"])

if n_done >= n_items:
    print(f"All {n_items} items already done -- nothing to generate.")
else:
    remaining_calls = (n_items - n_done) * calls_per_item - partial_done_calls
    print(
        f"{n_items - n_done} items left x {calls_per_item} calls "
        f"- {partial_done_calls} resumed calls = {remaining_calls} generation calls"
    )

    with tqdm(total=remaining_calls, desc="generating", unit="call") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < n_done:
                continue  # already checkpointed by an earlier run

            image = item["image"]
            transcription_messages = pilot.prompts.build_transcription_messages(image)
            grading_messages = pilot.prompts.build_grading_messages(image)

            if partial_state is not None and partial_state["item_idx"] == item_idx:
                entry = partial_state["entry"]
            else:
                entry = {
                    "item": {k: item[k] for k in META_FIELDS},
                    "transcription_samples_raw": [],
                    "grading_samples_raw": [],
                    "transcription_temp0_raw": [],
                    "grading_temp0_raw": [],
                }

            entry["transcription_samples_raw"] = run_batch(
                transcription_messages,
                K,
                True,
                TEMP,
                "transcribe T=0.7",
                item_idx,
                entry.setdefault("transcription_samples_raw", []),
                entry,
                pbar,
            )
            entry["grading_samples_raw"] = run_batch(
                grading_messages,
                K,
                True,
                TEMP,
                "grade T=0.7",
                item_idx,
                entry.setdefault("grading_samples_raw", []),
                entry,
                pbar,
            )
            entry["transcription_temp0_raw"] = run_batch(
                transcription_messages,
                N_TEMP0,
                False,
                None,
                "transcribe T=0",
                item_idx,
                entry.setdefault("transcription_temp0_raw", []),
                entry,
                pbar,
            )
            entry["grading_temp0_raw"] = run_batch(
                grading_messages,
                N_TEMP0,
                False,
                None,
                "grade T=0",
                item_idx,
                entry.setdefault("grading_temp0_raw", []),
                entry,
                pbar,
            )

            raw_results.append(entry)
            with open(completed_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()

            if os.path.exists(partial_path):
                os.remove(partial_path)
            partial_state = None

print(f"Collected raw samples for {len(raw_results)} items.")
print(f"Completed-item checkpoint on Drive: {completed_path}")
print(f"Partial checkpoint on Drive: {partial_path}")


Found completed-item checkpoint: 122 items
  /content/drive/MyDrive/uncertainty-math-vlm/checkpoints/raw_Qwen2.5-VL-3B-Instruct_n100_seed42.jsonl
Checkpoint item 24 does not match expected sample order; resuming from there.
Truncating completed checkpoint from 122 to 23 valid items.
Validated completed items: 23
77 items left x 14 calls - 0 resumed calls = 1078 generation calls


generating:   0%|          | 0/1078 [00:00<?, ?call/s]

item 24/100 | transcribe T=0.7 1/5
item 24/100 | transcribe T=0.7 2/5
item 24/100 | transcribe T=0.7 3/5
item 24/100 | transcribe T=0.7 4/5
item 24/100 | transcribe T=0.7 5/5
item 24/100 | grade T=0.7 1/5
item 24/100 | grade T=0.7 2/5
item 24/100 | grade T=0.7 3/5
item 24/100 | grade T=0.7 4/5
item 24/100 | grade T=0.7 5/5
item 24/100 | transcribe T=0 1/2
item 24/100 | transcribe T=0 2/2
item 24/100 | grade T=0 1/2
item 24/100 | grade T=0 2/2
item 25/100 | transcribe T=0.7 1/5
item 25/100 | transcribe T=0.7 2/5
item 25/100 | transcribe T=0.7 3/5
item 25/100 | transcribe T=0.7 4/5
item 25/100 | transcribe T=0.7 5/5
item 25/100 | grade T=0.7 1/5
item 25/100 | grade T=0.7 2/5
item 25/100 | grade T=0.7 3/5
item 25/100 | grade T=0.7 4/5
item 25/100 | grade T=0.7 5/5
item 25/100 | transcribe T=0 1/2
item 25/100 | transcribe T=0 2/2
item 25/100 | grade T=0 1/2
item 25/100 | grade T=0 2/2
item 26/100 | transcribe T=0.7 1/5
item 26/100 | transcribe T=0.7 2/5
item 26/100 | transcribe T=0.7 3/5
i

In [ ]:
# Checkpoint repair cell.
# Run this only if Cell 4 says the run is complete when you know it is not.
# It keeps the valid completed-item prefix and removes duplicate/bad rows.
import json
import os

N = 100
SEED = 42
K = 5
N_TEMP0 = 2
META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")

CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
completed_path = f"{CHECKPOINT_DIR}/raw_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}.jsonl"
partial_path = f"{CHECKPOINT_DIR}/raw_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}.partial.json"

sample = pilot.data.load_fermat_sample(n=N, seed=SEED)


def complete(entry):
    return (
        len(entry.get("transcription_samples_raw", [])) == K
        and len(entry.get("grading_samples_raw", [])) == K
        and len(entry.get("transcription_temp0_raw", [])) == N_TEMP0
        and len(entry.get("grading_temp0_raw", [])) == N_TEMP0
    )


def matches(entry, item):
    saved = entry.get("item", {})
    return all(saved.get(k) == item[k] for k in META_FIELDS)


if not os.path.exists(completed_path):
    raise FileNotFoundError(f"No completed checkpoint found: {completed_path}")

with open(completed_path) as f:
    entries = [json.loads(line) for line in f if line.strip()]

valid = []
for i, entry in enumerate(entries[:N]):
    if not complete(entry):
        print("First incomplete checkpoint item:", i + 1)
        break
    if not matches(entry, sample[i]):
        print("First mismatched checkpoint item:", i + 1)
        break
    valid.append(entry)

print("Original checkpoint rows:", len(entries))
print("Valid checkpoint rows:", len(valid))

with open(completed_path, "w") as f:
    for entry in valid:
        f.write(json.dumps(entry, default=str) + "\n")

# A stale partial checkpoint can also confuse resume after repair.
if os.path.exists(partial_path):
    os.remove(partial_path)
    print("Removed stale partial checkpoint:", partial_path)

print("Checkpoint repaired. Now rerun Cell 4 to continue.")


First mismatched checkpoint item: 24
Original checkpoint rows: 127
Valid checkpoint rows: 23
Checkpoint repaired. Now rerun Cell 4 to continue.


### Scoring cell — note on the temperature-0 anchor

`temp0_entropy_transcription` / `temp0_entropy_grading` below are computed over
**2** greedy draws per item, not 1. With only 1 sample, `cluster_entropy` would be
mathematically zero by definition regardless of anything the model actually did —
a tautology, not a finding. With 2 draws, a nonzero value is a real (if noisy)
signal of low-temperature instability — e.g. an accidentally-enabled sampling flag,
or batching nondeterminism — and should be read as a **pipeline smoke test**, not
as an empirical claim about the model's true low-temperature behavior. If these are
not at or near zero for nearly every item, something in the sampling or parsing is
broken and needs fixing before trusting anything else.

Per-item parse-failure counts (`n_transcription_parse_failures`,
`n_grading_parse_failures`, out of K=5) are also recorded here so Step 4 can check
the **aggregate** parse-failure rate across the whole run — a systematic regex or
prompt-format bug can make every sample in an item fail to parse, which collapses
to a single confident `<PARSE_FAILURE>` cluster (entropy 0) and would otherwise
hide inside numbers that look clean.

In [ ]:
# Scoring cell.
import importlib

import pilot.parsing
import pilot.entropy

importlib.reload(pilot.parsing)

scored_results = []
for entry in raw_results:
    item = entry["item"]

    transcription_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_samples_raw"]
    ]
    grading_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]
    transcription_temp0_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_temp0_raw"]
    ]
    grading_temp0_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_temp0_raw"]]

    perception_entropy = pilot.entropy.cluster_entropy(transcription_parsed)
    reasoning_entropy = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_parsed]
    )
    temp0_entropy_transcription = pilot.entropy.cluster_entropy(transcription_temp0_parsed)
    temp0_entropy_grading = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_temp0_parsed]
    )

    majority_transcription, _ = pilot.entropy.majority_cluster(transcription_parsed)
    transcription_correct = majority_transcription == pilot.entropy.normalize_string(
        item["pert_a"]
    )

    majority_grading, _ = pilot.entropy.majority_cluster(
        [None if d is None else str(d) for d in grading_parsed]
    )
    grading_correct = majority_grading in {"0", "1"} and int(majority_grading) == int(
    item["has_error"]
)

    n_transcription_parse_failures = sum(1 for t in transcription_parsed if t is None)
    n_grading_parse_failures = sum(1 for d in grading_parsed if d is None)

    scored_results.append(
        {
            "orig_q": item["orig_q"],
            "pert_a": item["pert_a"],
            "has_error": item["has_error"],
            "handwriting_style": item["handwriting_style"],
            "image_quality": item["image_quality"],
            "perception_entropy": perception_entropy,
            "reasoning_entropy": reasoning_entropy,
            "temp0_entropy_transcription": temp0_entropy_transcription,
            "temp0_entropy_grading": temp0_entropy_grading,
            "transcription_correct": transcription_correct,
            "grading_correct": grading_correct,
            "n_transcription_parse_failures": n_transcription_parse_failures,
            "n_grading_parse_failures": n_grading_parse_failures,
            "all_transcription_samples_raw": entry["transcription_samples_raw"],
            "all_grading_samples_raw": entry["grading_samples_raw"],
            "temp0_transcription_raw": entry["transcription_temp0_raw"],
            "temp0_grading_raw": entry["grading_temp0_raw"],
            "model_id": MODEL_ID,
            "n_items": N,
        }
    )

print(f"Scored {len(scored_results)} items.")

Scored 100 items.


In [ ]:
# Save cell: write CSV into the cloned repo's results/ dir, commit, push.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

df = pd.DataFrame(scored_results)

model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = f"results_{model_slug}_{timestamp}.csv"

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

# Write a copy to Drive BEFORE touching git. Pushing can fail for auth or
# fast-forward reasons, and losing an hour of GPU output to a git problem
# would be painful -- this copy survives regardless of what happens below.
drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup copy written to {drive_results}/{csv_name}")

# Secrets to scrub from any git output we print. A failed push makes git
# echo the remote URL back in its error message, which would otherwise
# print the token straight into this notebook's saved output.
_REDACT = []


def git(*args):
    """Run a git command in repo/, surfacing output (redacted) when it fails."""
    result = subprocess.run(
        ["git", "-C", "repo", *args], capture_output=True, text=True
    )
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


# A fresh clone has no committer identity, so `git commit` fails with exit
# 128 ("Please tell me who you are"). Set it for this clone only.
git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")

git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add pilot results: {csv_name}")
if commit.returncode != 0:
    raise RuntimeError("git commit failed -- see output above")
print(f"Committed {csv_name}")

# Pushing requires credentials even for a PUBLIC repo: anonymous HTTPS is
# read-only. Reuse the token from the auth cell if the repo was private,
# otherwise prompt for one now (needs 'repo' / contents:write scope).
GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")

    # The clone may be behind if anything was pushed from elsewhere since
    # this session started; rebase our new commit on top before pushing.
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")

    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print(
            "Push failed (see output above). The CSV is safe on Drive and in "
            "repo/results/ -- you can retry the push without re-running the model."
        )

Wrote repo/results/results_qwen25-vl-3b-instruct_20260731T210745Z.csv (100 rows)
Backup copy written to /content/drive/MyDrive/uncertainty-math-vlm/results/results_qwen25-vl-3b-instruct_20260731T210745Z.csv
Committed results_qwen25-vl-3b-instruct_20260731T210745Z.csv
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed (see output above). The CSV is safe on Drive and in repo/results/ -- you can retry the push without re-running the model.


In [ ]:
import pandas as pd

df = pd.DataFrame(scored_results)

print("Rows:", len(df))
print("Grading accuracy:", df["grading_correct"].mean())
print("Transcription accuracy:", df["transcription_correct"].mean())

print("\nReasoning entropy by grading correctness:")
print(
    df.groupby("grading_correct")["reasoning_entropy"]
    .agg(["count", "median", "mean", "min", "max"])
)

print("\nPerception entropy by transcription correctness:")
print(
    df.groupby("transcription_correct")["perception_entropy"]
    .agg(["count", "median", "mean", "min", "max"])
)

print("\nParse failures:")
print("Transcription:", df["n_transcription_parse_failures"].sum())
print("Grading:", df["n_grading_parse_failures"].sum())

Rows: 100
Grading accuracy: 0.75
Transcription accuracy: 0.0

Reasoning entropy by grading correctness:
                 count    median      mean  min       max
grading_correct                                          
False               25  0.500402  0.447328  0.0  0.950271
True                75  0.500402  0.341972  0.0  1.054920

Perception entropy by transcription correctness:
                       count    median      mean       min       max
transcription_correct                                               
False                    100  1.609438  1.531439  0.500402  1.609438

Parse failures:
Transcription: 67
Grading: 5


### Grading K-resample: does a higher K sharpen the reasoning-entropy signal?

K=5 grading entropy is coarse for what is effectively a binary label (the
parsed 0/1 error digit): only 3 realistic entropy values are reachable in
practice (unanimous 0, a 4-1 split at ~0.500, a 3-2 split at ~0.673), and on
the real 100-item run above the correct/incorrect groups have an *identical*
median (0.500402) at K=5 -- the observed AUROC=0.62 comes from the mean/tail,
not median separation. This experiment resamples **grading only** at K=25 to
test whether a less noisy entropy estimate breaks that tie or confirms the
signal is genuinely modest regardless of K.

Perception/transcription is untouched and does not need Cell 4 re-run -- the
5 already-collected T=0.7 grading samples per item are reused (not redrawn)
and combined with 20 new samples per item to reach K_GRADING=25.

**Prerequisite: Cells 1-3 (install, auth/clone/import, model load) must have
run this session. Do NOT run Cell 4 first** -- that would redundantly redo
the already-valid 1,400-call full pipeline; only new grading samples are
needed here.

The decision rule for what counts as "sharper" lives in
`pilot.plotting.classify_k_resample_result` and runs offline (no GPU) once
this CSV is downloaded locally -- see the plan for the exact thresholds.

In [ ]:
# Grading-only K-resample: setup and base-sample lookup.
import ast
import gc
import json
import os
import time

import pandas as pd
import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

N = 100
SEED = 42
K_BASE = 5              # already collected and analyzed this session (AUROC 0.62)
K_GRADING = 15           # <- new K under test; drop to 15 if this looks too slow
TEMP = 0.7
MAX_RETRIES = 3
RETRY_PAUSE_SECONDS = 5

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
# (orig_q, pert_a) is the join key everywhere below -- orig_q alone repeats
# across perturbed variants of the same question (91/100 unique on the real
# data), so it would silently misjoin rows. pilot.plotting.join_k_resample
# uses the same composite key for the offline before/after comparison.

INFRA_EXCEPTIONS = (
    ConnectionError,
    TimeoutError,
    torch.cuda.OutOfMemoryError,
    OSError,
)


def generate(messages, do_sample: bool, temperature: float | None):
    """Identical to Cell 4's generate() -- duplicated so this cell runs
    standalone without Cell 4 having executed first."""
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            text_prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text_prompt],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(model.device)

            gen_kwargs = {"max_new_tokens": 512, "do_sample": do_sample}
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.no_grad():
                output_ids = model.generate(**inputs, **gen_kwargs)

            trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
            return processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
        except INFRA_EXCEPTIONS as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_PAUSE_SECONDS)
    raise last_exc


def atomic_json_dump(obj, path):
    """Write JSON through a temp file so a crash mid-write cannot corrupt it."""
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f)
    os.replace(tmp_path, path)


sample = pilot.data.load_fermat_sample(n=N, seed=SEED)
n_items = len(sample)

CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BASE_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/raw_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}.jsonl"
# The K=5, AUROC=0.62 baseline run -- its push to GitHub failed (403
# permission error, visible in Cell 7's saved output above), so repo/results/
# is empty on a fresh clone; read the Drive backup instead.
BASELINE_CSV_NAME = "results_qwen25-vl-3b-instruct_20260731T210745Z.csv"


def load_base_grading_samples():
    """{(orig_q, pert_a): [K_BASE raw grading strings]}. Tries the raw
    checkpoint first, then the Drive results CSV backup. Items with no base
    found get a loud warning and draw the full K_GRADING fresh instead of
    silently assuming zero base samples.
    """
    lookup = {}
    if os.path.exists(BASE_CHECKPOINT_PATH):
        with open(BASE_CHECKPOINT_PATH) as f:
            entries = [json.loads(line) for line in f if line.strip()]
        for entry in entries:
            item = entry["item"]
            key = (item["orig_q"], item["pert_a"])
            raw = entry.get("grading_samples_raw", [])
            if len(raw) != K_BASE:
                continue
            if key in lookup:
                raise ValueError(f"Duplicate (orig_q, pert_a) key in base checkpoint: {key}")
            lookup[key] = raw
        print(f"Loaded {len(lookup)} base items from raw checkpoint: {BASE_CHECKPOINT_PATH}")

    baseline_csv_path = f"{DRIVE_RESULTS_DIR}/{BASELINE_CSV_NAME}"
    if len(lookup) < n_items and os.path.exists(baseline_csv_path):
        base_df = pd.read_csv(baseline_csv_path)
        for _, row in base_df.iterrows():
            key = (row["orig_q"], row["pert_a"])
            if key not in lookup:
                raw = ast.literal_eval(row["all_grading_samples_raw"])
                if len(raw) == K_BASE:
                    lookup[key] = raw
        print(f"Loaded {len(lookup)} base items (cumulative) after Drive CSV backup: {baseline_csv_path}")

    missing = n_items - len(lookup)
    if missing:
        print(
            f"WARNING: no K_BASE={K_BASE} baseline for {missing} item(s) -- "
            "drawing full K_GRADING fresh for those."
        )
    return lookup


base_lookup = load_base_grading_samples()

In [ ]:
# Grading-only K-resample: draw the additional samples, checkpointed per item
# PLUS a mid-item partial checkpoint every CHECKPOINT_EVERY_GENERATIONS calls,
# so a hang/disconnect partway through a single item's 20 calls doesn't lose
# all of them -- only up to CHECKPOINT_EVERY_GENERATIONS-1 calls, matching
# the safety net Cell 4 already has for the main pipeline.
CHECKPOINT_EVERY_GENERATIONS = 5

extra_checkpoint_prefix = f"grading_extra_k{K_GRADING}_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}"
extra_completed_path = f"{CHECKPOINT_DIR}/{extra_checkpoint_prefix}.jsonl"
extra_partial_path = f"{CHECKPOINT_DIR}/{extra_checkpoint_prefix}.partial.json"

extra_results = []
if os.path.exists(extra_completed_path):
    with open(extra_completed_path) as f:
        extra_results = [json.loads(line) for line in f if line.strip()]
    print(f"Found extra-sample checkpoint: {len(extra_results)} items -- {extra_completed_path}")

done_keys = {(e["item"]["orig_q"], e["item"]["pert_a"]) for e in extra_results}
remaining_items = [it for it in sample if (it["orig_q"], it["pert_a"]) not in done_keys]
print(f"{len(remaining_items)} of {n_items} items still need grading resampling at K={K_GRADING}.")

partial_state = None
if os.path.exists(extra_partial_path):
    with open(extra_partial_path) as f:
        candidate = json.load(f)
    if tuple(candidate.get("key", [])) not in done_keys:
        partial_state = candidate
        print(
            f"Found mid-item partial checkpoint: item {candidate['key']}, "
            f"{len(candidate['extra_raw'])} call(s) already saved."
        )
    else:
        print("Ignoring stale partial checkpoint for an already-completed item.")

total_new_calls = sum(
    K_GRADING - len(base_lookup.get((it["orig_q"], it["pert_a"]), []))
    for it in remaining_items
)

calls_since_partial_checkpoint = 0
with tqdm(total=total_new_calls, desc=f"grading resample K={K_GRADING}", unit="call") as pbar:
    for item in remaining_items:
        key = (item["orig_q"], item["pert_a"])
        base_raw = base_lookup.get(key, [])
        target_extra = K_GRADING - len(base_raw)  # 0 base samples -> draw all K_GRADING fresh
        grading_messages = pilot.prompts.build_grading_messages(item["image"])

        if partial_state is not None and tuple(partial_state["key"]) == key:
            extra_raw = partial_state["extra_raw"]
            pbar.update(len(extra_raw))  # already-done calls, counted in total_new_calls above
        else:
            extra_raw = []
        partial_state = None  # consumed (either used above or belonged to a different item)

        for j in range(len(extra_raw), target_extra):
            pbar.set_postfix_str(f"item {key[0][:24]!r} {j + 1}/{target_extra}")
            extra_raw.append(generate(grading_messages, do_sample=True, temperature=TEMP))
            pbar.update(1)
            calls_since_partial_checkpoint += 1
            if calls_since_partial_checkpoint >= CHECKPOINT_EVERY_GENERATIONS:
                atomic_json_dump({"key": list(key), "extra_raw": extra_raw}, extra_partial_path)
                calls_since_partial_checkpoint = 0

        entry = {
            "item": {k: item[k] for k in META_FIELDS},
            "grading_samples_extra_raw": extra_raw,
            "n_base_samples_reused": len(base_raw),
        }
        extra_results.append(entry)
        with open(extra_completed_path, "a") as f:
            f.write(json.dumps(entry, default=str) + "\n")
            f.flush()

        if os.path.exists(extra_partial_path):
            os.remove(extra_partial_path)

print(f"Collected extra grading samples for {len(extra_results)} items at K_GRADING={K_GRADING}.")
print(f"Checkpoint on Drive: {extra_completed_path}")

In [ ]:
# Grading-only K-resample: scoring.
import pilot.parsing
import pilot.entropy

extra_by_key = {(e["item"]["orig_q"], e["item"]["pert_a"]): e for e in extra_results}

scored_kresample = []
for item in sample:
    key = (item["orig_q"], item["pert_a"])
    base_raw = base_lookup.get(key, [])
    extra_entry = extra_by_key.get(key)
    combined_raw = base_raw + (extra_entry["grading_samples_extra_raw"] if extra_entry else [])
    if len(combined_raw) != K_GRADING:
        raise ValueError(f"Item {key} has {len(combined_raw)} samples, expected K_GRADING={K_GRADING}.")

    grading_parsed = [pilot.parsing.parse_grading(t) for t in combined_raw]
    grading_labels = [None if d is None else str(d) for d in grading_parsed]

    reasoning_entropy_new = pilot.entropy.cluster_entropy(grading_labels)
    majority_new, _ = pilot.entropy.majority_cluster(grading_labels)
    grading_correct_new = pilot.parsing.grading_cluster_matches_label(majority_new, item["has_error"])
    n_parse_failures_new = sum(1 for d in grading_parsed if d is None)

    scored_kresample.append(
        {
            "orig_q": item["orig_q"],
            "pert_a": item["pert_a"],
            "has_error": item["has_error"],
            "handwriting_style": item["handwriting_style"],
            "image_quality": item["image_quality"],
            f"reasoning_entropy_k{K_GRADING}": reasoning_entropy_new,
            f"grading_correct_k{K_GRADING}": grading_correct_new,
            f"n_grading_parse_failures_k{K_GRADING}": n_parse_failures_new,
            f"all_grading_samples_raw_k{K_GRADING}": combined_raw,
            "k_base": K_BASE,
            "k_grading": K_GRADING,
            "model_id": MODEL_ID,
            "n_items": N,
        }
    )

print(f"Scored {len(scored_kresample)} items at K_GRADING={K_GRADING}.")

In [ ]:
# Grading-only K-resample: save CSV, commit, push. Distinct filename --
# never overwrites the original 100-item baseline results file.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

df_kresample = pd.DataFrame(scored_kresample)

model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = f"grading_kresample_k{K_GRADING}_{model_slug}_{timestamp}.csv"

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df_kresample.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df_kresample)} rows)")

# Drive backup written BEFORE any git operation, same reasoning as Cell 7:
# a push can fail for auth/permission reasons (it did last time -- see Cell
# 7's saved output, a 403) and this copy survives regardless.
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
df_kresample.to_csv(f"{DRIVE_RESULTS_DIR}/{csv_name}", index=False)
print(f"Backup copy written to {DRIVE_RESULTS_DIR}/{csv_name}")

_REDACT = []


def git(*args):
    """Run a git command in repo/, surfacing output (redacted) when it fails."""
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add grading K-resample results (K={K_GRADING}): {csv_name}")
if commit.returncode != 0:
    raise RuntimeError("git commit failed -- see output above")
print(f"Committed {csv_name}")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is safe on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print(
            "Push failed (see output above -- if it says 403, your GH_TOKEN "
            "does not have write access to this repo; generate one with "
            "'repo' scope and set RESET_TOKENS=True in the auth cell). "
            "The CSV is safe on Drive and in repo/results/ either way."
        )